Read Bronze CSV from ADLS

Cast Data Types

Important because OpenSky CSV often stores numbers as strings.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

checkpoint_path2 = "abfss://real@uralibootcamp.dfs.core.windows.net/checkpoint/silver_flights_v4"
checkpoint_path_silver_delta = (
"abfss://real@uralibootcamp.dfs.core.windows.net/checkpoints/silver_delta"
)

bronze_stream = (
    spark.readStream
    .option("startingVersion", "0")
    .table("openskyreal.bronze.bronze_flights")
)

In [0]:
%sql
SELECT COUNT(*) FROM openskyreal.bronze.bronze_flights;

In [0]:
%sql
DESCRIBE DETAIL openskyreal.bronze.bronze_flights;

In [0]:
from pyspark.sql.functions import col

silver_df1 = (
    bronze_stream
    .withColumn("icao24", col("icao24").cast("string"))
    .withColumn("callsign", col("callsign").cast("string"))
    .withColumn("origin_country", col("origin_country").cast("string"))
    .withColumn("time_position", col("time_position").cast("timestamp"))
    .withColumn("last_contact", col("last_contact").cast("timestamp"))
    .withColumn("longitude", col("longitude").cast("double"))
    .withColumn("latitude", col("latitude").cast("double"))
    .withColumn("baro_altitude", col("baro_altitude").cast("double"))
    .withColumn("on_ground", col("on_ground").cast("boolean"))
    .withColumn("velocity", col("velocity").cast("double"))
    .withColumn("true_track", col("true_track").cast("double"))
    .withColumn("vertical_rate", col("vertical_rate").cast("double"))
    .withColumn("sensors", col("sensors").cast("string"))
    .withColumn("geo_altitude", col("geo_altitude").cast("double"))
    .withColumn("squawk", col("squawk").cast("string"))
    .withColumn("spi", col("spi").cast("boolean"))
    .withColumn("position_source", col("position_source").cast("integer"))
)

In [0]:
from pyspark.sql.functions import col, to_timestamp, from_unixtime

silver_df1 = (
    silver_df1

    # Create event_time from time_position
    .withColumn(
        "event_time",
        to_timestamp(
            from_unixtime(
                col("time_position").cast("long")
            )
        )
    )

    # Rename columns
    .withColumnRenamed(
        "baro_altitude",
        "altitude"
    )

)

Data Quality Cleaning

Remove invalid locations:

In [0]:
silver_df1 = (
    silver_df1
    .filter(
        (col("latitude").between(-90,90)) &
        (col("longitude").between(-180,180))
    )
)

In [0]:

silver_df1 = (
    silver_df1
    .withWatermark(
        "event_time",
        "10 minutes"
    )
    
)

Check NULL count for every column 

In [0]:
from pyspark.sql.functions import col, sum

null_check_df1 = silver_df1.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in silver_df1.columns
    ]
)

#display(null_check_df1, checkpointLocation=checkpoint_path2)

In [0]:
silver_df1 = silver_df1.drop("sensors")

Add Business Columns
Flight Speed Conversion

OpenSky velocity is meters/sec.

Convert to km/hour:

In [0]:
silver_df1 = silver_df1.withColumn(
    "speed_kmh",
    round(col("velocity") * 3.6,2)
)

Speed Category

For Flight Speed Analytics:

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "speed_category",
        when(col("speed_kmh") < 200,"Low")
        .when(col("speed_kmh") < 700,"Normal")
        .otherwise("High")
    )
)

Flight Direction Category

For air corridor analysis:

In [0]:
from pyspark.sql.functions import col, when

silver_df1 = (
    silver_df1
    .withColumn(
        "direction",
        when(
            (col("true_track") >= 0) &
            (col("true_track") < 90),
            "North-East"
        )
        .when(
            (col("true_track") >= 90) &
            (col("true_track") < 180),
            "South-East"
        )
        .when(
            (col("true_track") >= 180) &
            (col("true_track") < 270),
            "South-West"
        )
        .when(
            (col("true_track") >= 270) &
            (col("true_track") <= 360),
            "North-West"
        )
        .otherwise("Unknown")
    )
)

Aircraft Safety Anomaly Flags
Detect aircraft moving on ground

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "ground_movement_flag",
        when(
            (col("on_ground")==True) &
            (col("velocity") > 20),
            1
        )
        .otherwise(0)
    )
)

Detect unusual speed

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "speed_anomaly_flag",
        when(
            col("speed_kmh") > 1200,
            1
        )
        .otherwise(0)
    )
)

Air Traffic Density Region

Create geographic zones.

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "region",
        concat(
            round(col("latitude"),0),
            lit("_"),
            round(col("longitude"),0)
        )
    )
)

Add Processing Metadata

In [0]:
silver_df1 = (
    silver_df1
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

silver_clean_df1 = (
    silver_df1
    .filter(col("icao24").isNotNull())
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
    .withColumn("silver_timestamp", current_timestamp())
)

Write Silver Delta Table



Add Timestamp Columns (History Tracking)

Your Silver table should have:

In [0]:
silver_df1 = (
    silver_clean_df1
    .withColumn(
        "created_timestamp",
        current_timestamp()
    )
    .withColumn(
        "updated_timestamp",
        current_timestamp()
    )
    .withColumn(
        "effective_start_date",
        current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)


In [0]:
%sql 
Drop table if exists openskyreal.silver.flights;

DROP TABLE IF EXISTS openskyreal.silver.upsert;
    
DROP TABLE IF EXISTS openskyreal.silver.scd2_type;

In [0]:
%sql
SHOW TABLES IN openskyreal.silver;

In [0]:
dbutils.fs.rm(
    checkpoint_path_silver_delta,
    True
)

In [0]:

(
silver_df1.writeStream
.format("delta")
.outputMode("append")
.option(
    "checkpointLocation",checkpoint_path_silver_delta

)
.toTable("openskyreal.silver.flights")
)

In [0]:
%sql
SELECT COUNT(*) FROM openskyreal.silver.flights;

In [0]:
%sql
select * from openskyreal.silver.flights;

In [0]:
silver_path1 = (
"abfss://real@uralibootcamp.dfs.core.windows.net/silver/flights"
)


silver_query = (
    silver_df1.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path2
    )
    .option(
        "mergeSchema",
        "true"
    )
    .outputMode("append")
    .trigger(
        processingTime="2 minutes"
    )
    .option(
        "path",
        silver_path1
    )
    .start()
)

UPSERT using MERGE

Use case:

OpenSky sends the same aircraft again with updated:

location
speed
direction
altitude

In [0]:
from delta.tables import DeltaTable
from delta.tables import DeltaTable

def upsert_to_silver(microBatchDF, batchId):
    silver_table = DeltaTable.forPath(spark, silver_path1)

    (
        silver_table.alias("target")
        .merge(
            microBatchDF.alias("source"),
            """
            target.icao24 = source.icao24
            AND target.time_position = source.time_position
            """
        )
        .whenMatchedUpdate(
            set={
                "latitude": "source.latitude",
                "longitude": "source.longitude",
                "velocity": "source.velocity",
                "speed_kmh": "source.speed_kmh",
                "silver_load_timestamp": "current_timestamp()"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

(
    silver_df1.writeStream
    .foreachBatch(upsert_to_silver)
    .outputMode("append")  # merge requires "update" or "append", not "complete"
    .option("checkpointLocation", checkpoint_path_silver_delta)
    .start()
)





In [0]:

(
    silver_df1.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path_silver_delta)
    .toTable("openskyreal.silver.upsert")
)

In [0]:
%sql
select count(*) from openskyreal.silver.upsert

SCD Type 1 (Current Data)

Use when you only need latest aircraft status.
Aircraft ABC123

Old:
speed=700

New:
speed=850


Replace old value

MERGE handles this.

Use:

Silver_Current_Flights

Table:

icao24
latitude
longitude
speed
country
silver_load_timestamp

SCD Type 2 (History Tracking)

Use when you want flight movement history.

we can analyze:

flight route
movement pattern
congestion
anomaly

Rewrite Delta table with SCD columns

Because your table was created without these columns:

In [0]:
def upsert_scd(microBatchDF, batchId):
    silver_table = DeltaTable.forPath(spark, silver_path1)
    (
        silver_table.alias("target")
        .merge(
            microBatchDF.alias("source"),
            "target.icao24 = source.icao24 AND target.is_current = true"
        )
        .whenMatchedUpdate(set={"is_current": "false", "end_date": "current_timestamp()"})
        .whenNotMatchedInsertAll()
        .execute()
    )

(
    silver_df1.writeStream
    .foreachBatch(upsert_scd)
    .option("checkpointLocation", checkpoint_path_silver_delta)
    .start()
)

In [0]:
from delta.tables import DeltaTable
spark.sql(f"""
CREATE TABLE IF NOT EXISTS openskyreal.silver.scd2_type (
    icao24 STRING,
    callsign STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    velocity DOUBLE,
    speed_kmh DOUBLE,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current BOOLEAN,
    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
from delta.tables import DeltaTable

def upsert_scd2(microBatchDF, batchId):
    silver_scd_table = DeltaTable.forPath(spark,silver_path1)

    # Pass 1: close out current rows whose tracked values changed
    (
        silver_scd_table.alias("target")
        .merge(
            microBatchDF.alias("source"),
            """
            target.icao24 = source.icao24
            AND target.is_current = true
            """
        )
        .whenMatchedUpdate(
            condition="""
            target.latitude <> source.latitude
            OR target.longitude <> source.longitude
            OR target.velocity <> source.velocity
            """,
            set={
                "effective_end_date": "current_timestamp()",
                "is_current": "false",
                "updated_timestamp": "current_timestamp()"
            }
        )
        .execute()
    )

    # Pass 2: insert new current version for new keys AND for rows just closed above
    (
        silver_scd_table.alias("target")
        .merge(
            microBatchDF.alias("source"),
            """
            target.icao24 = source.icao24
            AND target.is_current = true
            """
        )
        .whenNotMatchedInsert(
            values={
                "icao24": "source.icao24",
                "callsign": "source.callsign",
                "latitude": "source.latitude",
                "longitude": "source.longitude",
                "velocity": "source.velocity",
                "speed_kmh": "source.speed_kmh",
                "effective_start_date": "current_timestamp()",
                "effective_end_date": "NULL",
                "is_current": "true",
                "created_timestamp": "current_timestamp()",
                "updated_timestamp": "current_timestamp()"
            }
        )
        .execute()
    )

In [0]:
dbutils.fs.rm(
    checkpoint_path_silver_delta,
    True
)

In [0]:

(
    silver_df1.writeStream
    .foreachBatch(upsert_scd2)
    .outputMode("update")
    .option("checkpointLocation", checkpoint_path_silver_delta)
    .start()
)

In [0]:
%sql
select count(*) from openskyreal.silver.upsert;

OPTIMIZE Silver Table

After many batch loads:

Purpose:

compact small files
improve query speed

In [0]:
%sql
DESCRIBE DETAIL openskyreal.silver.flights;

In [0]:
%sql
OPTIMIZE openskyreal.silver.flights;

VACUUM

Remove old unused files:

VACUUM opensky.silver.flights
RETAIN 168 HOURS;

History Tracking

Delta automatically keeps versions.

Check history:

In [0]:
%sql
DESCRIBE HISTORY openskyreal.silver.flights;

In [0]:
%sql
DESCRIBE HISTORY openskyreal.silver.upsert;

In [0]:
dbutils.fs.rm(
    checkpoint_path_silver_delta,
    True
)

In [0]:
%sql
DESCRIBE HISTORY openskyreal.silver.scd2_type;

Time Travel

You can query previous versions:

In [0]:
%sql
SELECT *
FROM openskyreal.silver.flights
VERSION AS OF 0;

In [0]:
%sql 
select * from openskyreal.silver.upsert;

In [0]:
%sql 
select * from openskyreal.silver.scd2_type;